# Serial and Parallel Agent Workflows with CrewAI

Two ways to organize the same agents:

- **Serial:** a researcher gathers sourced findings, then a writer turns them into an executive summary. The writer depends on the researcher.
- **Parallel:** two researchers investigate independent questions *at the same time*, then a writer merges both into one briefing.

The notebook records a **timeline** of every search call and task completion, so you can see whether the parallel branches really overlap rather than taking it on faith.

**Requirements:** an Anthropic API key and a Tavily API key.


## Serial vs. parallel: what's actually different

```
SERIAL                                        PARALLEL (fan-out / fan-in)

Researcher ──notes──▶ Writer ──▶ result       Drug-discovery researcher ─┐
                                                                         ├──▶ Writer ──▶ combined result
(step 2 can't start until step 1 is done)     Diagnostics researcher ────┘
                                              (both start at once; writer waits for both)
```

| | Serial | Parallel |
|---|---|---|
| **Use it when** | A later step needs an earlier step's output | Subtasks are independent of each other |
| **Data flow** | Each task reads the previous task's output | Branches share nothing; a final step merges them |
| **Wall-clock time** | Sum of all steps | Roughly the slowest branch, plus the merge step |
| **Token cost** | Same work, same tokens | Same work, same tokens. Parallelism saves time, not money |
| **Failure behavior** | One failed step blocks everything after it | One failed branch can be retried without redoing the other |
| **Main risk** | Errors flow downstream into later steps | Branches can quietly duplicate or contradict each other; the merge step has to reconcile them |
| **Rate limits** | One request at a time | Concurrent requests hit API rate limits sooner |
| **CrewAI setting** | Default: tasks run in list order, and each task automatically receives all previous outputs as context | `async_execution=True` on each branch, `context=[]` so branches don't wait on each other, and a final task with `context=[branch_1, branch_2]` |

### The three CrewAI details that make or break parallelism
1. **Context is inherited by default.** Unless you set `context` explicitly, a task receives every earlier task's output, which also means it **waits** for them. In the original version of this notebook that silently turned the "parallel" run back into a serial one.
2. **Only the last task's output is returned.** `crew.kickoff()` returns the final task's result. Without a merge step, the other branch's work is still computed but disappears from the result.
3. **A crew can't end on an async task.** CrewAI requires the crew to finish with at most one asynchronous task, which is another reason the fan-in step belongs at the end.

## Setup

In [1]:
%pip install -q "crewai[anthropic]" requests nest_asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46

In [6]:
import os, re, time, asyncio

import requests
import nest_asyncio
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool

nest_asyncio.apply()  # notebook event loops need this for CrewAI's async kickoff


def load_secret(name: str) -> str:
    """Read a secret from Colab Secrets, falling back to an environment variable."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = os.environ.get(name)
    if not value:
        raise RuntimeError(f"Missing {name}. Add it to Colab Secrets or set it as an environment variable.")
    return value

os.environ["ANTHROPIC_API_KEY"] = load_secret("ANTHROPIC_API_KEY")
os.environ["TAVILY_API_KEY"] = load_secret("TAVILY_API_KEY")

MODEL = os.environ.get("ANTHROPIC_MODEL", "anthropic/claude-sonnet-5-5")
VERBOSE = False  # True shows CrewAI's step-by-step panels; set False before saving for GitHub
MAX_SUMMARY_WORDS = 100

llm = LLM(model=MODEL, max_tokens=3000)

## Instrumented search tool
Each tool instance carries a `label` so the timeline can show *which branch* made each search call. Every result is also stored in `SOURCE_LOG`, which the citation check uses as the record of what the agents actually saw.

In [7]:
SOURCE_LOG: dict[str, dict] = {}   # url -> {"title", "content"}
TIMELINE: list[tuple] = []         # (seconds since run start, label, event)
RUN_START = [time.perf_counter()]

def log_event(label: str, event: str):
    TIMELINE.append((time.perf_counter() - RUN_START[0], label, event))

def start_run():
    TIMELINE.clear()
    RUN_START[0] = time.perf_counter()


class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = (
        "Search the web for recent information. Returns numbered results with title, URL "
        "and an excerpt of the page content. Input: a search query string."
    )
    label: str = "search"

    def _run(self, query: str) -> str:
        log_event(self.label, f"search: {query[:60]}")
        response = requests.post(
            "https://api.tavily.com/search",
            headers={"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"},
            json={"query": query, "max_results": 5, "search_depth": "advanced"},
            timeout=30,
        )
        response.raise_for_status()
        results = response.json().get("results", [])
        if not results:
            return "No web results found."
        blocks = []
        for i, r in enumerate(results, start=1):
            url, content = r.get("url", ""), (r.get("content") or "").strip()[:1500]
            SOURCE_LOG[url] = {"title": r.get("title", ""), "content": content}
            blocks.append(f"[{i}] {r.get('title', '')}\nURL: {url}\nExcerpt: {content}")
        return "\n\n".join(blocks)


def show_timeline(title: str):
    print(f"\n{title}")
    print(f"{'time':>7}  {'branch':<18} event")
    for t, label, event in sorted(TIMELINE):
        print(f"{t:6.1f}s  {label:<18} {event}")

## Shared rules and checks
Every agent must cite a source URL for each claim. After each run, `check_citations` confirms that every cited URL was actually retrieved during the run.

In [8]:
CITATION_RULES = (
    "Use ONLY facts that appear in your search excerpts. End every factual sentence with "
    "(Source: <exact URL from the search results>). If you can't source something, leave it out."
)
URL_PATTERN = re.compile(r"https?://[^\s)\]>]+")

def word_count(text: str) -> int:
    text = URL_PATTERN.sub("", text)
    text = re.sub(r"\(Source:\s*\)", "", text)
    return len(re.findall(r"[A-Za-z0-9$%][\w$%.,'’\-/]*", text))

def word_limit_guardrail(output):
    """CrewAI task guardrail: send the summary back if it is over the limit."""
    n = word_count(output.raw)
    if n > MAX_SUMMARY_WORDS:
        return (False, f"The summary is {n} words; the limit is {MAX_SUMMARY_WORDS} (URLs excluded). Shorten it and keep the citations.")
    return (True, output)

def check_citations(label: str, text: str):
    cited = {u.rstrip(".,;") for u in URL_PATTERN.findall(text)}
    unknown = sorted(u for u in cited if u not in SOURCE_LOG)
    status = "PASS" if cited and not unknown else "FLAG"
    detail = f"{len(cited)} cited, all retrieved" if status == "PASS" else (
        "no citations" if not cited else f"not retrieved: {', '.join(unknown)}")
    print(f"{status}  {label:<28} {detail}")

## Part 1: Serial workflow
The writer **needs** the researcher's findings, so these tasks must run in order. CrewAI's default sequential process does exactly that, and `context=[research_task]` makes the dependency explicit.

In [14]:
serial_researcher = Agent(
    role="AI Healthcare Researcher",
    goal="Find the 3 most significant recent advancements in AI for healthcare, backed by sources you retrieved.",
    backstory="A careful analyst who only reports what the sources say.",
    tools=[TavilySearchTool(label="researcher")],
    llm=llm, max_iter=10, allow_delegation=False, verbose=VERBOSE,
)

writer = Agent(
    role="Executive Technical Writer",
    goal="Turn sourced research notes into short, accurate executive summaries.",
    backstory="You never add facts that aren't in the notes, and you keep every citation attached to its claim.",
    tools=[],  # no search: the writer can only use what it's handed
    llm=llm, allow_delegation=False, verbose=VERBOSE,
)

research_task = Task(
    description=("Research the top 3 recent advancements in AI for healthcare. Run several searches and "
                 "write 2-4 sentences on each advancement. " + CITATION_RULES),
    expected_output="Notes on three advancements; every sentence ends with (Source: URL).",
    agent=serial_researcher,
    callback=lambda out: log_event("researcher", "TASK DONE"),
)

summary_task = Task(
    description=(f"Write an executive summary of the research notes in at most {MAX_SUMMARY_WORDS} words "
                 "(URLs excluded): exactly 3 bullet points, one per advancement, each ending with "
                 "(Source: <URL from the notes>). Add nothing that isn't in the notes."),
    expected_output=f"3 cited bullets, at most {MAX_SUMMARY_WORDS} words.",
    agent=writer,
    context=[research_task],   # explicit dependency: this is what makes the workflow serial
    guardrail=word_limit_guardrail,
    callback=lambda out: log_event("writer", "TASK DONE"),
)

serial_crew = Crew(agents=[serial_researcher, writer], tasks=[research_task, summary_task], verbose=VERBOSE)

start_run()
serial_result = await serial_crew.kickoff_async()
serial_seconds = time.perf_counter() - RUN_START[0]

print(serial_result.raw)
show_timeline(f"Serial timeline ({serial_seconds:.1f}s wall-clock)")
print()
check_citations("Serial summary", serial_result.raw)
print(f"Word count: {word_count(serial_result.raw)} / {MAX_SUMMARY_WORDS}")

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:250: UserWarning: function callbacks cannot be serialized and will prevent checkpointing. Use a module-level named function instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


- **Ambient AI scribes:** A JAMA study across five academic medical centers found they cut clinician documentation time by 16 minutes per visit. (Source: https://www.2minutemedicine.com/ambient-artificial-intelligence-scribes-reduce-clinician-documentation-time-by-16-minutes-per-encounter)

- **AI-discovered drugs:** A 2024 Drug Discovery Today analysis found 80–90% Phase I success, substantially higher than historic industry averages. Phase II success was about 40% on a limited sample size, comparable to historic averages. (Source: https://pubmed.ncbi.nlm.nih.gov/38692505)

- **FDA-authorized AI devices:** One tracker analysis reports 1,451 AI/ML devices authorized through 2025, with 295 cleared in 2025 alone. (Source: https://intuitionlabs.ai/articles/fda-ai-medical-device-tracker)

Serial timeline (20.5s wall-clock)
   time  branch             event
   2.4s  researcher         search: recent advancements AI healthcare 2025
   2.4s  researcher         search: ambient AI scribes clini

In the timeline, every writer event comes **after** the researcher's `TASK DONE`. That's the defining property of a serial workflow.

## Part 2: Parallel workflow (fan-out / fan-in)
Two questions that don't depend on each other:
- **Branch A:** which AI companies are leading in drug discovery?
- **Branch B:** how is AI changing patient diagnostics?

Each branch gets **its own agent** (sharing one agent across concurrent tasks invites state conflicts), `async_execution=True`, and `context=[]` so it doesn't wait for anything. A final **merge task** receives both outputs, which fixes the lost-output problem.

In [15]:
def make_parallel_workflow(parallel: bool):
    """Build the fan-out/fan-in crew. parallel=False runs the same tasks serially, for a timing baseline."""
    drug_researcher = Agent(
        role="Drug Discovery Analyst",
        goal="Identify AI companies leading in drug discovery, backed by sources you retrieved.",
        backstory="An industry analyst who reports only what sources state.",
        tools=[TavilySearchTool(label="A: drug discovery")],
        llm=llm, max_iter=10, allow_delegation=False, verbose=VERBOSE,
    )
    dx_researcher = Agent(
        role="Clinical Diagnostics Analyst",
        goal="Explain how AI is changing patient diagnostics, backed by sources you retrieved.",
        backstory="A clinical technology analyst who reports only what sources state.",
        tools=[TavilySearchTool(label="B: diagnostics")],
        llm=llm, max_iter=10, allow_delegation=False, verbose=VERBOSE,
    )
    merge_writer = Agent(
        role="Executive Technical Writer",
        goal="Merge independent research streams into one concise briefing.",
        backstory="You never add facts that aren't in the notes, and you keep every citation attached to its claim.",
        tools=[], llm=llm, allow_delegation=False, verbose=VERBOSE,
    )

    branch_a = Task(
        description=("List 5 AI companies focused on drug discovery, with one line each on what they specialize in. "
                     + CITATION_RULES),
        expected_output="5 companies, one cited line each.",
        agent=drug_researcher,
        async_execution=parallel,
        context=[],   # independent: don't wait for or inherit anything
        callback=lambda out: log_event("A: drug discovery", "TASK DONE"),
    )
    branch_b = Task(
        description=("Describe 3 concrete ways AI is changing patient diagnostics, with a real example of each. "
                     + CITATION_RULES),
        expected_output="3 cited points with examples.",
        agent=dx_researcher,
        async_execution=parallel,
        context=[],   # without this, CrewAI would hand branch A's output to branch B, and make B wait for it
        callback=lambda out: log_event("B: diagnostics", "TASK DONE"),
    )
    merge = Task(
        description=("Combine both research notes into one executive briefing with two headed sections: "
                     "'AI in Drug Discovery' (the companies) and 'AI in Diagnostics' (the 3 points). "
                     "Keep every (Source: URL) citation. Add nothing that isn't in the notes. Under 200 words."),
        expected_output="A two-section cited briefing.",
        agent=merge_writer,
        context=[branch_a, branch_b],   # fan-in: waits for both branches, receives both outputs
        callback=lambda out: log_event("merge", "TASK DONE"),
    )
    return Crew(agents=[drug_researcher, dx_researcher, merge_writer],
                tasks=[branch_a, branch_b, merge], verbose=VERBOSE)


start_run()
parallel_result = await make_parallel_workflow(parallel=True).kickoff_async()
parallel_seconds = time.perf_counter() - RUN_START[0]
parallel_timeline = list(TIMELINE)

print(parallel_result.raw)
show_timeline(f"Parallel timeline ({parallel_seconds:.1f}s wall-clock)")
print()
check_citations("Parallel briefing", parallel_result.raw)

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:250: UserWarning: function callbacks cannot be serialized and will prevent checkpointing. Use a module-level named function instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


## AI in Drug Discovery

- **Recursion Pharmaceuticals:** "TechBio" company using AI, machine learning and high-throughput automation; runs the Recursion Operating System. (Source: https://ca.finance.yahoo.com/news/recursion-pharmaceuticals-inc-rxrx-advances-211544210.html)
- **Insilico Medicine:** Clinical-stage, generative-AI-driven; its Pharma.AI platform spans biology, generative chemistry, clinical medicine and research. (Source: https://finance.yahoo.com/news/insilico-medicine-secures-110-million-130000028.html)
- **Isomorphic Labs:** Google DeepMind spin-off; its IsoDDE is a unified drug-design system going beyond AlphaFold 3. (Source: https://www.isomorphiclabs.com/articles/the-isomorphic-labs-drug-design-engine-unlocks-a-new-frontier)
- **Schrödinger:** Physics-based platform for molecular modeling and simulation, including FEP+. (Source: https://intuitionlabs.ai/software/drug-discovery-research/computational-chemistry/schrodinger)
- **insitro:** Machine learning-enabled; comb

### Did the branches really overlap?
If the workflow is truly parallel, branch B's first search should happen **before** branch A finishes. This cell checks that from the timeline instead of eyeballing the log.

In [11]:
def overlap_report(timeline):
    def first(label, event_prefix):
        times = [t for t, l, e in timeline if l == label and e.startswith(event_prefix)]
        return min(times) if times else None
    def done(label):
        times = [t for t, l, e in timeline if l == label and e == "TASK DONE"]
        return max(times) if times else None

    a_start, a_done = first("A: drug discovery", "search"), done("A: drug discovery")
    b_start, b_done = first("B: diagnostics", "search"), done("B: diagnostics")
    if None in (a_start, a_done, b_start, b_done):
        print("Not enough events to judge overlap (a branch may not have searched).")
        return
    overlap = max(0.0, min(a_done, b_done) - max(a_start, b_start))
    verdict = "PARALLEL: the branches overlapped" if overlap > 0 else "SERIAL: one branch finished before the other started"
    print(f"Branch A active {a_start:5.1f}s to {a_done:5.1f}s")
    print(f"Branch B active {b_start:5.1f}s to {b_done:5.1f}s")
    print(f"Overlap: {overlap:.1f}s  ->  {verdict}")

overlap_report(parallel_timeline)

Branch A active   2.4s to  17.4s
Branch B active   2.9s to  19.3s
Overlap: 14.5s  ->  PARALLEL: the branches overlapped


### Optional: timing baseline
Run the exact same three tasks **serially** to measure what parallelism saved. This repeats every API call, so it roughly doubles the cost of the notebook; it's off by default.

In [18]:
RUN_SERIAL_BASELINE = True

if RUN_SERIAL_BASELINE:
    start_run()
    await make_parallel_workflow(parallel=False).kickoff_async()
    baseline_seconds = time.perf_counter() - RUN_START[0]
    overlap_report(list(TIMELINE))
    print(f"\nSame tasks, serial:   {baseline_seconds:6.1f}s")
    print(f"Same tasks, parallel: {parallel_seconds:6.1f}s")
    print(f"Time saved:           {baseline_seconds - parallel_seconds:6.1f}s "
          f"({(1 - parallel_seconds / baseline_seconds):.0%})")

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:250: UserWarning: function callbacks cannot be serialized and will prevent checkpointing. Use a module-level named function instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


Branch A active   6.1s to  15.3s
Branch B active  18.2s to  32.9s
Overlap: 0.0s  ->  SERIAL: one branch finished before the other started

Same tasks, serial:     52.8s
Same tasks, parallel:   30.7s
Time saved:             22.2s (42%)


## Sources retrieved
Every page the agents were shown across both runs, for manual spot-checking.

In [19]:
for url, src in SOURCE_LOG.items():
  print(f"- {src['title']}\n  {url}")

- AI Drug Discovery Clinical Trials: 173 Programs Hit Milestones | MapleLine Ventures
  https://maplelineventures.com/fr/blog/ai-drug-discovery-reaches-clinical-trial-milestones-173-programs-enter-critical-phase
- AI Drug Discovery Clinical Trials: 173 Programs Hit Milestones | MapleLine Ventures
  https://maplelineventures.com/blog/ai-drug-discovery-reaches-clinical-trial-milestones-173-programs-enter-critical-phase
- How to View Clinical Milestones in AI Drug Discovery｜Pharma Insight Lab
  https://note.com/pharma_insight/n/n642dacd56e01?hl=en
- RXRX: AI-driven drug discovery platform advances with key partnerships and upcoming clinical milestones — TradingView News
  https://www.tradingview.com/news/urn:summary_document_transcript:quartr.com:3102191:0-rxrx-ai-driven-drug-discovery-platform-advances-with-key-partnerships-and-upcoming-clinical-milestones
- AI-enabled drug discovery reaches clinical milestone | Semantic Scholar
  https://www.semanticscholar.org/paper/AI-enabled-drug-dis

## Takeaways
- **Parallel saves time, not tokens.** Both workflows do the same amount of model work.
- **Fan-out needs a fan-in.** Without a merge step, CrewAI returns only the last task's output.

## Limitations
- The citation check confirms URLs were retrieved, not that each sentence is faithful to its source.
- Timings vary between runs with API latency; run the baseline a few times before quoting a speed-up.
- Concurrency is limited by your API rate limits; many parallel branches can trigger throttling.